# Week 7B — Regularisation

Regularisation discourages overly complex models. This notebook explores two ways to constrain decision trees and introduces Ridge and Lasso penalties for linear regression.

## Learning goals
- Explain why a model may overfit.
- Apply `max_depth` and `min_samples_leaf` to decision trees.
- Compare Ridge (L2) and Lasso (L1) regularisation.
- Explain why features should be standardised before coefficient penalties.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso

RANDOM_STATE = 42

## 2. Load the student dataset

In [ ]:
# Load the course dataset (run this notebook from its notebooks/ folder)
df = pd.read_csv("../data/student_pass_fail.csv")
df.head()

## 3. Train/test split

In [ ]:
X = df[["study_hours", "attendance"]]
y = df["passed"]

# Same split as the decision-tree notebook (same seed and settings)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

## 4. Regularising decision trees

A tree can be constrained with:
- `max_depth`: limit the number of levels
- `min_samples_leaf`: every leaf must contain at least this many training students

These limits can reduce overfitting, although a simpler tree is not guaranteed to perform better on every split.

In [ ]:
# Each entry: (label, settings passed to DecisionTreeClassifier)
tree_settings = [
    ("Unrestricted", {}),
    ("Depth 3", {"max_depth": 3}),
    ("Depth 5, leaf size 5", {"max_depth": 5, "min_samples_leaf": 5}),
    ("Depth 3, leaf size 10", {"max_depth": 3, "min_samples_leaf": 10})
]

tree_rows = []
for name, settings in tree_settings:
    model = DecisionTreeClassifier(random_state=RANDOM_STATE, **settings)
    model.fit(X_train, y_train)
    tree_rows.append({
        "Model": name,
        "Leaves": model.get_n_leaves(),               # fewer leaves = simpler tree
        "Training accuracy": model.score(X_train, y_train),
        "Test accuracy": model.score(X_test, y_test)
    })

pd.DataFrame(tree_rows).round(3)

**Look for:** the unrestricted tree has the most leaves and the highest training accuracy. Does it also have the highest test accuracy?

## 5. Ridge and Lasso: penalising coefficients

For linear regression, regularisation adds a penalty to the data-fitting loss:

**Objective = loss + alpha × penalty**

- **Ridge (L2):** penalises squared coefficients and shrinks them all towards zero.
- **Lasso (L1):** penalises absolute coefficients and can set some coefficients **exactly to zero** (feature selection).

`passed` is binary, so it is not a regression target. For this part we switch to a second dataset, `student_exam_scores.csv`: 300 students, eight features and a continuous **exam score** (0–100). This is not real data, just a practice toy.

| Feature | Meaning |
|---|---|
| `study_hours` | hours studied per week |
| `attendance` | lecture attendance (%) |
| `assignments_done` | assignments submitted (out of 10) |
| `prior_grade` | grade in the previous module |
| `sleep_hours` | average sleep per night |
| `phone_hours` | daily phone screen time |
| `commute_min` | commute to campus (minutes) |
| `shoe_size` | shoe size (EU) |

Two of these features have **no real effect** on the exam score. Can you guess which? A good regularised model should learn to ignore them.

In [ ]:
# Load the regression dataset (same folder as the pass/fail data)
exam = pd.read_csv("../data/student_exam_scores.csv")
display(exam.head())

feature_cols = ["study_hours", "attendance", "assignments_done", "prior_grade",
                "sleep_hours", "phone_hours", "commute_min", "shoe_size"]
X_reg = exam[feature_cols]
y_reg = exam["exam_score"]

# 75% training, 25% test (no stratify: the target is continuous)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=RANDOM_STATE
)
print("Training rows:", len(Xr_train), "| Test rows:", len(Xr_test))

## 6. Compare ordinary linear regression, Ridge, and Lasso

The features are **standardised inside each pipeline**. The penalty depends on coefficient size, and coefficient size depends on a feature's units, so without scaling the penalty would treat features unfairly.

`alpha` controls the penalty strength; larger values mean stronger regularisation.

> scikit-learn scales the Ridge and Lasso objectives differently, so the same `alpha` number does not mean the same strength in both. Compare each model with plain linear regression, not alpha with alpha.

In [ ]:
models = {
    "Linear regression": make_pipeline(StandardScaler(), LinearRegression()),
    "+Ridge (alpha=50)": make_pipeline(StandardScaler(), Ridge(alpha=50)),
    "+Lasso (alpha=1)": make_pipeline(StandardScaler(), Lasso(alpha=1, max_iter=10000))
}

score_rows, coefficient_rows = [], []
for name, model in models.items():
    model.fit(Xr_train, yr_train)
    # R² = share of the target's variation explained (1 = perfect)
    score_rows.append({
        "Model": name,
        "Training R²": model.score(Xr_train, yr_train),
        "Test R²": model.score(Xr_test, yr_test)
    })
    # model[-1] is the last pipeline step (the regression model); coef_ holds its coefficients
    for feature, coef in zip(X_reg.columns, model[-1].coef_):
        coefficient_rows.append({"Model": name, "Feature": feature, "Coefficient": coef})

display(pd.DataFrame(score_rows).round(3))
coef_df = pd.DataFrame(coefficient_rows)
coef_pivot = coef_df.pivot(index="Feature", columns="Model", values="Coefficient")
coef_pivot = coef_pivot[list(models)]   # keep column order: linear, Ridge, Lasso
display(coef_pivot.round(3))

In [ ]:
ax = coef_pivot.loc[feature_cols[::-1]].plot(kind="barh", figsize=(9, 6))   # keep the feature order top to bottom
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Coefficient (standardised features)")
ax.set_ylabel("")
ax.set_title("Ridge shrinks every coefficient; Lasso sets some exactly to zero")
plt.tight_layout()
plt.show()

**Look for:**
- With standardised features, each coefficient is the change in exam score for a one-standard-deviation increase in that feature. Which features matter most? Which one lowers the score?
- Ridge makes every coefficient smaller, but none becomes zero.
- Lasso sets `commute_min` and `shoe_size` to exactly 0. These are the two features with no real effect, so it has removed them.
- Test R² is only slightly lower: the penalties simplify the model without losing much accuracy.

## 7. Investigate the penalty strength

Run this cell and compare how Ridge and Lasso coefficients change as `alpha` increases (note the different alpha ranges, for the reason given above).

In [ ]:
# Separate alpha ranges, because the two objectives are scaled differently
ridge_alphas = np.logspace(-2, 4.5, 40)
lasso_alphas = np.logspace(-2, 1.2, 40)

ridge_path = [make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(Xr_train, yr_train)[-1].coef_
              for a in ridge_alphas]
lasso_path = [make_pipeline(StandardScaler(), Lasso(alpha=a, max_iter=20000)).fit(Xr_train, yr_train)[-1].coef_
              for a in lasso_alphas]
ridge_path, lasso_path = np.asarray(ridge_path), np.asarray(lasso_path)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for j, feature in enumerate(feature_cols):
    style = "--" if feature in ["commute_min", "shoe_size"] else "-"   # dashed = no real effect
    axes[0].plot(ridge_alphas, ridge_path[:, j], style, label=feature)
    axes[1].plot(lasso_alphas, lasso_path[:, j], style, label=feature)
for ax, title in zip(axes, ["Ridge (L2)", "Lasso (L1)"]):
    ax.set_xscale("log")
    ax.set_xlabel("alpha (stronger penalty →)")
    ax.set_title(title)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.grid(alpha=0.2)
axes[0].set_ylabel("Coefficient (standardised features)")
axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), title="dashed = no real effect")
plt.tight_layout()
plt.show()

**Look for:** Ridge curves approach zero smoothly but never reach it; Lasso curves hit zero one after another. The no-effect features drop out first, then the weakest real ones (`sleep_hours`, `assignments_done`), and `study_hours` survives longest.

## Key takeaways

- Tree regularisation (`max_depth`, `min_samples_leaf`) limits the complexity of decision rules.
- Ridge shrinks all coefficients; Lasso can set coefficients exactly to zero.
- Standardise features before using coefficient penalties.
- Larger alpha → simpler model. Choosing alpha properly needs data the final test set doesn't touch (cross-validation, Week 10).